In [ ]:
# Transformer 모델 구축 - Transformer RAG(Retriever Augmentation Generation) 구성 및 모델 파이프라인 구축(데이터 수집기 + Qdrant 의미 기반 검색엔진 적용)
# - Retrieval 단계: Qdrant 의미 기반 검색엔진을 통해 관련 문서를 빠르게 찾아냄
# - Augmentation 단계: 검색된 문서를 기반으로 LLM 입력 프롬프트를 강화 → 맥락 있는 답변 생성
# - Generation 단계: LLM이 최종 답변을 생성 → 단순 생성보다 정확성과 신뢰성이 높아짐
# - 예시) 구글, 네이버에서 사용되는 RAG 시스템 구현

# 학습 목표 - 실무에서 사용되는 파이프라인 이해 및 적용
# 1. 데이터 수집기
# - 수집 대상 도메인: Google News RSS
# - RSS 피드 파싱: feedparser 라이브러리로 RSS XML을 읽고 기사 항목 추출
# - 데이터 정제: title,link,description,pubDate,soure 필드 추출, pubDate -> Python datetime 변화
# - DB 저장: PostgreSQL 테이블 news_articles 에 Insert
# - 수집 데이터 -> PostgreSQL 테이블 생성 및 입력
# - DB 조회 + 로깅 설정으로 데이터 관리

# 2. Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

# 3. 임베딩 생성 후 Qdrant Collection에 Insert/Update
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 4. Qdrant 의미 기반 검색
# - 의미 기반 검색으로 관련 문서 조회

# 5. QA 처리
# - Qdrant 검색 결과 -> QA 토크나이저 + QA 모델
# - 형태소 분석으로 한국어 처리 강화
# - 문자열 -> 토큰 ID 변환, 예시: "AI는 의료 분야에서 활용된다." -> [101, 1234, 5678, ...]
# - 모델 입력: input_ids 토큰 ID 배열, attention_mask 패딩 여부 표시, 모델 출력을 텍스트로 복원 [101, 1234, 5678, ...] -> "AI는 의료 분야에서 활용된다."

# 6. 요약 처리
# - 요약 토크나이저 + 요약 모델 (KoBART 기반)
# - 반복 억제, 길이 조절, 다양성 확보 등 파라미터 튜닝
# - 후처리(clean_summary)로 중복 제거
# - from transformers import BartTokenizer # 영어 전용이라 맞지 않음

# 7. 최종 응답: 외부 서비스 연계 검토
# - Local LLM은 GPU 장비 한계로 로컬 실행은 패스 -> 대신 Copilot에 문의하여 자연스러운 답변 재구성
# - 검토: 현재 로컬 GPU 장비로는 생성형 LLM 모델을 도메인에 맞추어 파인튜닝은 불가능, 현재 추론 모델도 좋은 성능의 모델로 교체 불가능

# 8. 서비스: 구글, 네이버 RAG 시스템 구성
# - /llm_app/transformer_rag2_23_app.py
# - FastAPI 구동 정보: 터미널에서 구동, uvicorn transformer_rag2_23_app:app --reload, 경로 포함 uvicorn LLM.llm_app.transformer_rag2_23_app:app --reload
# - FastAPI 서비스: /search, 입력: 질의문, 출력: QA + 요약 결과 + 출처 정보
# - 1)  [사용자 질의]
# - 2)  [FastAPI 엔드포인트: /search]
# - 3)  [SentenceTransformer: 임베딩]
# - 4)  [Qdrant 의미 기반 검색]
# - 5)  [검색 결과 문서]
# - 6)  [KoELECTRA QA 모델 + MeCab 후처리(형태소 분석: 한국어 처리 강황)]
# - 7)  [KoBART Summarization 모델 + clean_summary]
# - 8)  [응답 + 출처 표시]
# - 9)  [최종 응답 LLM 모델은  외부 서비스 연계 검토: 자연스러운 문장]
# - 10) [최종 사용자 응답]

In [7]:
# 데이터 수집기: Google News RSS
import feedparser
from bs4 import BeautifulSoup
from datetime import datetime
import psycopg2
import logging

# 로깅 설정: 파일 저장
logging.basicConfig(
    level=logging.INFO, # 로그 레벨: INFO 이상 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 로그 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 로그를 파일에 저장
        logging.StreamHandler() # 콘솔에도 출력
    ]
)

# DB 연결 한번만 수행
conn = psycopg2.connect(
    dbname="newsdb",      # 생성한 DB 이름
    user="newsuser",      # 생성한 사용자
    password="1234",      # 설정한 비밀번호
    host="localhost",     # 로컬에서 실행 중
    port="5432"           # 기본 포트
)
cur = conn.cursor()

# Google News RSS -> DB Insert
def insert_article(title, content, url, published_at, source_name, source_url):
    try: # 중복 뉴스는 무시되고, 새로운 뉴스만 저장: ON CONFLICT (url) url 컬럼 값이 이미 존재하면 충동 발생, DO NOTHING 충돌시 아무것도 하지 않고 넘어감
        cur.execute("""
            INSERT INTO news_articles (title, content, url, published_at, source_name, source_url)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING;
        """, (title, content, url, published_at, source_name, source_url))
        if cur.rowcount > 0: # 새로운 행 추가시 1
            logging.info("데이터 Insert 성공: %s", title)
        else: # 새로운 행 추가 없을시 0
            logging.info("중복으로 건너뜀: %s", title)
        conn.commit()
    except Exception as e:
        logging.error("Insert 실패: %s, 에러: %s", title, e)

# Google News RSS
rss_url = "https://news.google.com/rss?hl=ko&gl=KR&ceid=KR:ko"
feed = feedparser.parse(rss_url)

for entry in feed.entries:
    title = entry.title # title
    
    # description HTML 제거
    content_raw = entry.description if "description" in entry else ""
    content = BeautifulSoup(content_raw, "html.parser").get_text()

    url = entry.link # url
    # pubDate 처리
    if hasattr(entry, "published_parsed"): # hasattr() 객체에 안전한 속성 접근을 위해 사용
        published_at = datetime(*entry.published_parsed[:6]) # feedparser가 RSS <pubDate> 태그를 자동으로 읽어와서 속성으로 제공
    else:
        published_at = datetime.now()
    # source 처리
    if hasattr(entry, "source"):
        source_name = entry.source.title # 언론사 이름(본문 텍스트)
        source_url = entry.source.href # 언론사 URL(속성)
    else:
        source_name = "Google News"
        source_url = ""

    # DB Insert 함수
    insert_article(title, content, url, published_at, source_name, source_url)
    # print(f"{title}\n{content}\n{url}\n{published_at}\n{source_name}\n{source_url}") # 데이터 확인

# 마지막 연결 객체 닫기: 순서 지켜서 닫아야 함
cur.close()
conn.close()
logging.info("DB 연결 종료 완료")

2026-04-02 13:20:37,188 [INFO] 중복으로 건너뜀: 트럼프 “석유 필요하면 미국산 사라…이란 2~3주 내 초강력 타격” - 한겨레
2026-04-02 13:20:37,191 [INFO] 중복으로 건너뜀: 홍준표, 김부겸 지지 선언... “대구 도움 될 행정가 뽑아야” - 조선일보
2026-04-02 13:20:37,194 [INFO] 중복으로 건너뜀: ‘종전’한다면서…세번째 미 항모 중동으로 출발 - 경향신문
2026-04-02 13:20:37,199 [INFO] 중복으로 건너뜀: “이란, 호르무즈 통과 유조선에 배럴당 1달러 통행료”…위안화·코인 결제 - 경향신문
2026-04-02 13:20:37,199 [INFO] 중복으로 건너뜀: 경유지라더니… 정원오 일행 칸쿤 일정, 해변·박물관 관광으로 채워져 - 조선일보
2026-04-02 13:20:37,204 [INFO] 중복으로 건너뜀: "54년 만 달 비행, 韓 위성도 탔다"···아르테미스 2호 발사 성공 - 헬로디디
2026-04-02 13:20:37,207 [INFO] 중복으로 건너뜀: 트럼프 “한국, 도움 안 돼”…주한미군 또 부풀리며 ‘파병 불참’ 비난 - 한겨레
2026-04-02 13:20:37,207 [INFO] 중복으로 건너뜀: '캐리어 시신' 피해여성 살해·유기한 20대 부부 오늘 구속심사 - 연합뉴스
2026-04-02 13:20:37,215 [INFO] 중복으로 건너뜀: 8일부터 공공기관 차량 2부제·공영주차장 5부제 시행 - 대한민국 정책브리핑
2026-04-02 13:20:37,217 [INFO] 중복으로 건너뜀: 민주당, 식사 자리서 금품 제공한 김관영 전북지사 제명···“국민께 송구” - 경향신문
2026-04-02 13:20:37,223 [INFO] 중복으로 건너뜀: '美서 나면 미국민' 뒤집은 트럼프…대법 '직관'에도 승소불투명(종합2보) - 연합뉴스
2026-04-02 13:20:37,226 [INFO] 중복으로 건너뜀: 트럼프 “종전 뒤 나토 탈퇴? 그

In [8]:
# Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + Google News RSS 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models # Qdrant 컬렉션 및 검색 관련 설정 정의하는 데이터 모델(구조체 클래스) 로드
# Qdrant 클라이언트 연결: Qdrant 검색엔진 서버 실행 및 서버 연결 확인
qdrant = QdrantClient(host="localhost", port=6333)

# Qdrant 컬렉션 생성
# - 벡터 크기는 모델에 따라 다르다(sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 -> 384차원)
# - 거리(metric)는 보통 COSINE 사용
# 컬렉션 존재 여부 확인 후 생성
if not qdrant.collection_exists("news_articles_collection"):
    qdrant.create_collection(
        collection_name="news_articles_collection",
        vectors_config=models.VectorParams(
            size=384, # 임베딩 벡터 차원: paraphrase-multilingual-MiniLM-L12-v2 임베딩 모델과 차원을 동일하게 맞추어야 한다
            distance=models.Distance.COSINE # 유사도 계산 방식
        )
    )
    print("Qdrant 컬렉션 생성 완료")
else:
    print("news_articles_collection 컬렉션이 존재합니다.")

2026-04-02 13:25:41,391 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-04-02 13:25:43,476 [INFO] HTTP Request: GET http://localhost:6333/collections/news_articles_collection/exists "HTTP/1.1 200 OK"
2026-04-02 13:25:46,168 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles_collection "HTTP/1.1 200 OK"


Qdrant 컬렉션 생성 완료
